<a href="https://colab.research.google.com/github/bug-request/benchmark/blob/scope/notebooks/scope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --index-url https://europe-west1-python.pkg.dev/bug-request/packages/simple/ benchmark-notebook


Looking in indexes: https://europe-west1-python.pkg.dev/bug-request/packages/simple/


In [ ]:
import plotly.express as px
import polars as pl

from google.colab import auth
from benchmark_notebook.io import CloudStorage

In [ ]:
auth.authenticate_user()

In [ ]:
storage = CloudStorage(bucket="bug-request-bucket")

df = pl.from_arrow(storage.asset("scope-repository-profile", location="local")).filter(
    ~pl.col("license").is_in({
      None,
      "other",
      "gpl-2.0",
      "gpl-3.0",
      "lgpl-2.1",
      "lgpl-3.0",
      "agpl-3.0",
      "mpl-2.0",
      "epl-2.0",
      "eupl-1.2",
      "cc-by-sa-4.0",
  })
)

df

name,total_changes,total_merges,total_contributors,total_authors,total_reviews,total_reviewers,total_issues,total_resolutions,median_reviews_per_change,median_reviewers_per_change,review_share,audit_only_share,label_share,default_target_share,license,language,creation,window_days,throughput,core_ratio,quality_score
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,str,str,str,i64,f64,f64,f64
"""sparcs-kaist/taxi-front""",2,1,1,1,2,2,3,1,1,1,1.0,0.0,1.0,1.0,"""mit""","""TypeScript""","""2021-03-16T12:46:23Z""",1,1.791759,0.5,1.0
"""ROCm/TheRock""",13,8,3,2,34,5,3,2,3,2,0.5,0.076923,0.0,1.0,"""mit""","""Python""","""2024-03-01T08:56:06Z""",1,4.836282,0.6,0.0
"""langgenius/dify-official-plugi…",3,4,1,2,2,2,2,1,1,1,0.75,0.0,1.0,1.0,"""apache-2.0""","""Python""","""2024-12-18T09:20:44Z""",1,2.995732,0.333333,0.75
"""prisma/prisma""",5,4,2,1,5,3,2,1,1,1,0.75,0.0,1.0,1.0,"""apache-2.0""","""TypeScript""","""2019-06-20T13:33:47Z""",1,3.401197,0.666667,0.75
"""ProjectLibertyLabs/siwf-embedd…",9,4,2,0,14,2,4,4,2,1,0.75,0.0,0.0,1.0,"""apache-2.0""","""TypeScript""","""2025-06-23T17:23:10Z""",1,3.912023,1.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""huggingface/transformers""",9,12,4,3,2,2,5,3,0,0,0.5,0.0,1.0,1.0,"""apache-2.0""","""Python""","""2018-10-29T13:56:00Z""",1,4.867534,0.571429,0.5
"""Foundryborne/daggerheart""",2,3,0,2,2,2,7,3,1,1,0.666667,0.0,1.0,0.666667,"""mit""","""JavaScript""","""2025-05-22T14:52:53Z""",1,2.484907,0.0,0.444444
"""ray-project/ray""",16,14,5,7,5,3,4,7,0,0,0.428571,0.0625,1.0,0.9375,"""apache-2.0""","""Python""","""2016-10-25T19:38:30Z""",1,5.541264,0.416667,0.376674


In [ ]:
df.columns

['name',
 'total_changes',
 'total_merges',
 'total_contributors',
 'total_authors',
 'total_reviews',
 'total_reviewers',
 'total_issues',
 'total_resolutions',
 'median_reviews_per_change',
 'median_reviewers_per_change',
 'review_share',
 'audit_only_share',
 'label_share',
 'default_target_share',
 'license',
 'language',
 'creation',
 'window_days']

In [ ]:
list(df.unique("license").select("license").to_series())

['apache-2.0',
 'zlib',
 'mit-0',
 'cc0-1.0',
 'unlicense',
 'cc-by-4.0',
 'mit',
 'isc',
 'bsd-3-clause-clear',
 'mulanpsl-2.0',
 'upl-1.0',
 'bsd-2-clause',
 'bsd-3-clause']

In [ ]:
df.select(pl.col("language")).group_by("language").len().sort("len", descending=True)

language,len
str,u32
"""Python""",1150
"""TypeScript""",977
"""JavaScript""",264


In [ ]:
px.box(df, x="label_share", color="language")

In [ ]:
px.histogram(df, x="creation", color="language")

In [ ]:
px.box(df, x="median_reviews_per_change", color="language")

In [ ]:
px.box(df, x="median_reviewers_per_change", color="language")

In [ ]:
df = df.with_columns(
    (
        (pl.col("total_changes") / pl.col("window_days")).log1p()
        + (pl.col("total_merges") / pl.col("window_days")).log1p()
    ).alias("throughput")
)
df = df.with_columns(
    (
        pl.col("total_contributors")
        / (pl.col("total_contributors") + pl.col("total_authors")).clip(lower_bound=1)
    ).alias("core_ratio")
)
df = df.with_columns(
    (
        pl.col("label_share")
        * pl.col("review_share")
        * pl.col("default_target_share")
        * (1-pl.col("audit_only_share"))
    ).alias("quality_score")
)

In [ ]:
import polars as pl
import plotly.express as px

factors = [
    "label_share",
    "review_share",
    "default_target_share",
    "audit_only_share",
    "quality_score",
]

px.box(
    df.unpivot(
        index=["language"],
        on=factors,
        variable_name="factor",
        value_name="value",
    ),
    x="factor",
    y="value",
    color="language",
    points="outliers",
).update_layout(
    xaxis_title="Factor",
    yaxis_title="Value",
    boxmode="group",
).show()

In [ ]:
df.filter(pl.col("quality_score") > 0.75)

name,total_changes,total_merges,total_contributors,total_authors,total_reviews,total_reviewers,total_issues,total_resolutions,median_reviews_per_change,median_reviewers_per_change,review_share,audit_only_share,label_share,default_target_share,license,language,creation,window_days,throughput,core_ratio,quality_score
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,str,str,str,i64,f64,f64,f64
"""department-of-veterans-affairs…",585,252,78,11,1629,82,1,1,2,1,0.785714,0.010256,1.0,0.973684,"""cc0-1.0""","""JavaScript""","""2015-03-31T18:34:05Z""",30,5.261135,0.876404,0.757191
"""snowflakedb/snowpark-python""",197,49,14,5,762,17,5,3,4,2,0.77551,0.015228,1.0,1.0,"""apache-2.0""","""Python""","""2021-04-19T18:24:15Z""",30,2.992003,0.736842,0.7637
"""google/osv.dev""",96,65,6,3,189,5,12,4,1,1,0.784615,0.010417,1.0,1.0,"""apache-2.0""","""Python""","""2021-01-08T02:14:10Z""",30,2.587764,0.666667,0.776442
"""MapChillE/uble-fe""",89,45,3,0,127,3,39,32,2,1,0.844444,0.044944,0.96875,1.0,"""apache-2.0""","""TypeScript""","""2025-07-08T02:34:48Z""",30,2.294217,1.0,0.781289
"""awslabs/mcp""",266,71,25,29,1249,40,48,32,4,2,0.887324,0.0,0.96875,1.0,"""apache-2.0""","""Python""","""2025-03-21T00:39:00Z""",30,3.503085,0.462963,0.859595
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""napari/napari""",88,27,7,6,327,8,27,18,2,2,0.888889,0.0,1.0,1.0,"""bsd-3-clause""","""Python""","""2018-08-13T01:12:28Z""",30,2.011341,0.538462,0.888889
"""keras-team/keras""",50,29,7,4,136,8,11,17,2,1,0.827586,0.02,1.0,0.961538,"""apache-2.0""","""Python""","""2015-03-28T00:35:42Z""",30,1.657169,0.636364,0.779841
"""Kernel360/KFE3_e2e_n-much""",60,29,5,0,112,5,31,28,2,2,0.896552,0.0,0.964286,1.0,"""mit""","""TypeScript""","""2025-06-12T11:03:14Z""",30,1.774952,1.0,0.864532


In [ ]:
px.box(df.filter(pl.col("quality_score") > 0.75), y="quality_score", color="language")

In [ ]:
px.scatter(df.filter(pl.col("quality_score") > 0.75), x="throughput", y="core_ratio", color="language")

In [ ]:
px.box(df,x="quality_score", color="language")

In [ ]:
px.box(df,x="throughput", color="language")

In [ ]:
order = ["low", "mid", "high"]

px.density_heatmap(
    df.filter(pl.col("quality_score") > 0.75).with_columns(
        pl.col("throughput").qcut(3, labels=order).alias("throughput_bin"),
        pl.col("core_ratio").qcut(3, labels=order).alias("core_bin"),
    )
    .group_by(["core_bin", "throughput_bin"])
    .len()
    .rename({"len": "count"}),
    x="throughput_bin",
    y="core_bin",
    z="count",
    text_auto=True,
    category_orders={"throughput_bin": order, "core_bin": order},
).update_layout(
    width=600,
    height=600,
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
).update_yaxes(autorange="reversed").show()

In [ ]:
px.scatter(df, y="quality_score", x="creation", color="language")